In [1]:
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "plotting":
    REPO_ROOT = REPO_ROOT.parent

import csv
import json
from matplotlib import pyplot as plt

plt.rcParams["font.family"] = "Charter"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Charter"
plt.rcParams["mathtext.it"] = "Charter:italic"
plt.rcParams["mathtext.bf"] = "Charter:bold"

SERIES_ORDER = ("ippo", "mappo", "pimac_v0", "pimac_v6")
ALGORITHM_LABELS = {
    "ippo": "IPPO",
    "mappo": "MAPPO",
    "pimac_v0": "PIC-MAPPO",
    "pimac_v6": "PC3D",
}
ALGORITHM_COLORS = {
    "ippo": "royalblue",
    "mappo": "#005d5d",
    "pimac_v0": "goldenrod",
    "pimac_v6": "#9f1853",
}
AXIS_LABEL_FONTSIZE = 13
LOW_BOXPLOT_MARKER_SIZE = 70

OUTPUT_DIR = REPO_ROOT / "plotting" / "boxplots"

PLOTS_TO_MAKE = [
    "spread_final_selected",
    "lbf_final_selected",
    "rware_final_selected",
]

DPI = 300
COMPRESS_LOW_OUTLIERS = True
LOW_OUTLIER_GAP_FRACTION = 0.55

SELECTED_CONFIGS = {
    "lbf_final_selected": {
        "title": "LBF",
        "output_filename": "lbf_hard_final_learning_curves.png",
        "task_config_path": "lbf_hard/task.json",
        "results_root": "results/lbf_hard",
        "run_prefix": "final_lbf_hard",
        "ippo": "best_01",
        "mappo": "best_01",
        "pimac_v0": "best_01",
        "pc3d": "active_01",
    },
    "rware_final_selected": {
        "title": "RWARE",
        "output_filename": "rware_final_learning_curves.png",
        "task_config_path": "robotic_warehouse_dynamic/task.json",
        "results_root": "results/rware",
        "run_prefix": "final_robotic_warehouse_dynamic",
        "ippo": "best_01",
        "mappo": "best_01",
        "pimac_v0": "best_01",
        "pc3d": "active_01",
    },
    "spread_final_selected": {
        "title": "Spread",
        "output_filename": "spread_hard_final_learning_curves.png",
        "task_config_path": "simple_spread_dynamic_hard/task.json",
        "results_root": "results/spread",
        "run_prefix": "final_simple_spread_dynamic_hard",
        "ippo": "best_01",
        "mappo": "best_01",
        "pimac_v0": "best_01",
        "pc3d": "active_03",
    },
}


In [2]:
def build_series(selected):
    results_root = selected["results_root"].rstrip("/")
    run_prefix = selected["run_prefix"]
    configs = {
        "ippo": selected["ippo"],
        "mappo": selected["mappo"],
        "pimac_v0": selected["pimac_v0"],
        "pimac_v6": selected["pc3d"],
    }
    return [
        {
            "label": ALGORITHM_LABELS[algorithm],
            "color": ALGORITHM_COLORS[algorithm],
            "glob": f"{results_root}/{algorithm}/{run_prefix}_{algorithm}_{configs[algorithm]}_s*/train_history.csv",
        }
        for algorithm in SERIES_ORDER
    ]


def boxplot_output_path(selected):
    base = Path(selected["output_filename"])
    return OUTPUT_DIR / f"{base.stem}_final_eval_boxplots{base.suffix}"


def load_task_count_split_labels(path):
    task_config = json.loads(path.read_text(encoding="utf-8"))
    labels = {}
    for count in task_config.get("train_counts", []):
        labels[int(count)] = "seen"
    for count in task_config.get("validation_counts", []):
        labels[int(count)] = "unseen/val"
    for count in task_config.get("test_counts", []):
        labels[int(count)] = "unseen/test"
    return labels


def load_final_eval_means(eval_path):
    rows = list(csv.DictReader(eval_path.open("r", encoding="utf-8")))
    return {
        int(row["n_agents"]): float(row["return_mean"])
        for row in rows
        if row.get("phase") == "final_checkpoint_test"
    }


def coarse_split_label(label):
    return "seen" if label == "seen" else "unseen"


def should_compress_low_boxplot(values, reference_values):
    reference_min = min(reference_values)
    reference_max = max(reference_values)
    reference_range = max(reference_max - reference_min, 1e-9)
    return max(values) < reference_min - LOW_OUTLIER_GAP_FRACTION * reference_range


def add_low_out_of_range_marker(axis, position, y_min, y_max, color):
    marker_y = y_min + 0.025 * (y_max - y_min)
    axis.scatter(
        [position],
        [marker_y],
        marker="v",
        s=LOW_BOXPLOT_MARKER_SIZE,
        color=color,
        edgecolor="#222222",
        linewidth=0.35,
        alpha=0.95,
        clip_on=False,
        zorder=4,
    )


In [3]:
def plot_boxplots(preset_name):
    selected = SELECTED_CONFIGS[preset_name]
    series = build_series(selected)
    split_labels = load_task_count_split_labels(REPO_ROOT / selected["task_config_path"])
    output_path = boxplot_output_path(selected)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    values_by_count_by_series = {}
    for series_index, item in enumerate(series):
        run_paths = sorted(REPO_ROOT.glob(item["glob"]))
        series_eval_means = [load_final_eval_means(path.with_name("eval_by_count.csv")) for path in run_paths]
        series_counts = set.intersection(*(set(eval_means) for eval_means in series_eval_means))
        for count in sorted(series_counts):
            values_by_count_by_series.setdefault(count, [[] for _ in series])
            values_by_count_by_series[count][series_index] = [eval_means[count] for eval_means in series_eval_means]

    counts = sorted(
        count for count, grouped_values in values_by_count_by_series.items()
        if all(len(values) > 0 for values in grouped_values)
    )

    fig, axes = plt.subplots(
        nrows=1,
        ncols=len(counts),
        figsize=(2 * len(counts), 3.2),
        dpi=DPI,
        sharey=False,
        squeeze=False,
    )

    for axis, count in zip(axes.flat, counts):
        split_label = coarse_split_label(split_labels.get(count, "unseen"))
        grouped_values = values_by_count_by_series[count]
        reference_values = [
            value
            for item, values in zip(series, grouped_values)
            if item["label"] != "IPPO"
            for value in values
        ]

        compressed_markers = []
        plotted_values = []
        plotted_positions = []
        plotted_series = []
        for position, (item, values) in enumerate(zip(series, grouped_values), start=1):
            compress = (
                COMPRESS_LOW_OUTLIERS
                and item["label"] == "IPPO"
                and should_compress_low_boxplot(values, reference_values)
            )
            if compress:
                compressed_markers.append((position, item["color"]))
                continue
            plotted_values.append(values)
            plotted_positions.append(position)
            plotted_series.append(item)

        flat_values = [value for values in plotted_values for value in values]
        y_min = min(flat_values)
        y_max = max(flat_values)
        y_pad = max(1e-6, 0.08 * (y_max - y_min if y_max > y_min else max(abs(y_min), 1.0)))
        y_min -= y_pad
        y_max += y_pad

        boxplot = axis.boxplot(
            plotted_values,
            positions=plotted_positions,
            patch_artist=True,
            widths=0.65,
            medianprops={"color": "#222222", "linewidth": 1.4},
            whiskerprops={"color": "#444444", "linewidth": 1.0},
            capprops={"color": "#444444", "linewidth": 1.0},
            boxprops={"linewidth": 1.0, "color": "#444444"},
            flierprops={"marker": "o", "markersize": 3, "markerfacecolor": "#666666", "markeredgewidth": 0.0, "alpha": 0.5},
        )
        for patch, item in zip(boxplot["boxes"], plotted_series):
            patch.set_facecolor(item["color"])
            patch.set_alpha(0.55)

        axis.set_ylim(y_min, y_max)
        for position, color in compressed_markers:
            add_low_out_of_range_marker(axis, position, y_min, y_max, color)

        axis.set_title(f"n={count} ({split_label})", fontsize=AXIS_LABEL_FONTSIZE * 1.25, fontweight="bold")
        axis.set_xticks(range(1, len(series) + 1))
        axis.set_xticklabels([item["label"] for item in series], rotation=25, ha="right", fontsize=12)
        axis.tick_params(axis="x", pad=0)
        axis.grid(True, axis="y", alpha=0.18)

    axes.flat[0].set_ylabel("Mean evaluation returns", fontsize=AXIS_LABEL_FONTSIZE * 1.25)
    fig.tight_layout(rect=(0.012, 0.060, 0.997, 0.905), pad=0.45, w_pad=0.70, h_pad=0.35)
    fig.text(0.5, 0.0, "Algorithm", ha="center", va="bottom", fontsize=AXIS_LABEL_FONTSIZE * 1.25)
    fig.savefig(output_path, bbox_inches="tight", dpi=DPI)
    plt.close(fig)
    return output_path


In [4]:
saved_paths = [plot_boxplots(preset_name) for preset_name in PLOTS_TO_MAKE]

for path in saved_paths:
    print(path)


/Users/akman/pimac/plotting/boxplots/spread_hard_final_learning_curves_final_eval_boxplots.png
/Users/akman/pimac/plotting/boxplots/lbf_hard_final_learning_curves_final_eval_boxplots.png
/Users/akman/pimac/plotting/boxplots/rware_final_learning_curves_final_eval_boxplots.png
